## Path config
Update paths before run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
repo_url = 'https://github.com/Batyaoftheyear/short-video-music-recommendation.git'
clone_dir = '/content/short-video-music-recommendation'
project_root = clone_dir
artifacts_dir = '/content/drive/MyDrive/short-video-music-recommendation/artifacts/retrieval_colab'

video_features = 'features/video_features.csv'
audio_features = 'features/audio_features_trimmed.csv'
train_triplets = 'features/pairs/train_triplets.csv'
video_train_split = 'features/splits/video_train.csv'
video_val_split = 'features/splits/video_val.csv'
audio_train_split = 'features/splits/audio_train.csv'
audio_val_split = 'features/splits/audio_val.csv'
val_relevance = 'reports/review/val_relevance_final.csv'

In [ ]:
from pathlib import Path

if not Path(clone_dir).exists():
    !git clone {repo_url} {clone_dir}
else:
    %cd {clone_dir}
    !git pull

%cd {clone_dir}
Path(artifacts_dir).mkdir(parents=True, exist_ok=True)
print('Project root:', project_root)
print('Artifacts dir:', artifacts_dir)

In [ ]:
from pathlib import Path
if Path('requirements.txt').exists():
    !pip -q install -r requirements.txt
elif Path('requirements_features.txt').exists():
    !pip -q install -r requirements_features.txt
else:
    !pip -q install torch pandas numpy scikit-learn joblib

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
required = [
    video_features, audio_features, train_triplets,
    video_train_split, video_val_split,
    audio_train_split, audio_val_split,
    val_relevance,
]
missing = [p for p in required if not Path(p).exists()]
if missing:
    raise FileNotFoundError('Missing required files:
' + '
'.join(missing))
print('All required files are present.')

## Train

In [ ]:
!python scripts/train_retrieval_model.py \
  --video-features {video_features} \
  --audio-features {audio_features} \
  --train-triplets {train_triplets} \
  --video-train-split {video_train_split} \
  --video-val-split {video_val_split} \
  --audio-train-split {audio_train_split} \
  --audio-val-split {audio_val_split} \
  --val-relevance {val_relevance} \
  --output-dir {artifacts_dir}

## Evaluate

In [ ]:
eval_output = f'{artifacts_dir}/eval_val'
!python scripts/evaluate_retrieval.py \
  --checkpoint {artifacts_dir}/best.pt \
  --video-features {video_features} \
  --audio-features {audio_features} \
  --video-split {video_val_split} \
  --audio-split {audio_val_split} \
  --relevance-file {val_relevance} \
  --preproc-dir {artifacts_dir} \
  --output-dir {eval_output}

In [ ]:
import json
from pathlib import Path
best_ckpt = Path(artifacts_dir) / 'best.pt'
metrics_path = Path(artifacts_dir) / 'eval_val' / 'metrics.json'
print('Best checkpoint:', best_ckpt)
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    print('Validation metrics:')
    print(json.dumps(metrics, indent=2, ensure_ascii=False))
else:
    print('metrics.json not found at', metrics_path)